# NB07 — Complete artifact→pathology shortcut audit

NB05 quantified only the LIGHT→EROSION pair. This notebook generalizes the audit to
**every {SALIVA, LIGHT} × {5 pathologies} pair**, with no retraining: it loads the
five `swin_m2_l06` fold checkpoints and pools the test predictions across all folds
(the full n=1,990).

For each (artifact, pathology) pair we report:
- **co-occurrence** — P(artifact=1 | pathology=1) on the pooled test set;
- **F1 with vs. without artifact** — pathology F1 on the subset where the artifact is
  present vs. absent (a large positive gap suggests the artifact *helps* → possible shortcut);
- **FP-rate under artifact-only** — fraction of images with the artifact present but the
  pathology absent that get a false positive (the direct shortcut signal);
- **bootstrap CI** on the FP-rate, so we know which signals are real vs. noise.

## 0. Installation and imports

In [1]:
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import f1_score

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens...')
        !unzip -q /content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip -d /content/
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/anonymous-V3/Trilha3-V2')
    IMGS_DIR = Path('e:/anonymous-V3/Data/Imgs')

SPLITS_DIR   = BASE_DIR / 'splits'
CKPT_DIR     = BASE_DIR / 'checkpoints'
RESULTS_DIR  = BASE_DIR / 'results' / 'nb07'
FIGS_DIR     = BASE_DIR / 'figs' / 'training'
NB03_RESULTS = BASE_DIR / 'results' / 'm2' / 'm2_results.json'  # thresholds por fold

for d in [RESULTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
ARTIFACTS   = ['SALIVA', 'LUZ']
N_CLASSES   = len(CORE_LABELS)
N_FOLDS     = 5
IMG_SIZE    = 224
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EXP_NAME     = 'm2_l06'
CKPT_PATTERN = f'swin_{EXP_NAME}_fold{{fold}}.pt'

print(f'Device      : {DEVICE}')
print(f'Modelo      : {EXP_NAME}')
print(f'Artefatos   : {ARTIFACTS}')
print(f'Patologias  : {CORE_LABELS}')

## 2. Dataset, model and per-fold prediction (no retraining)

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
val_transform = T.Compose([
    T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.imgs_dir = imgs_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def build_swin_tiny(n_classes=N_CLASSES):
    return timm.create_model('swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
                             pretrained=False, num_classes=n_classes)

def load_fold_model(fold):
    ckpt = CKPT_DIR / CKPT_PATTERN.format(fold=fold)
    assert ckpt.exists(), f'Checkpoint não encontrado: {ckpt}'
    model = build_swin_tiny()
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model

@torch.no_grad()
def predict_fold(model, df):
    ds = GastroscopyDataset(df, IMGS_DIR, val_transform)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    probs = []
    for imgs, _ in dl:
        probs.append(torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy())
    return np.concatenate(probs)

print('Funções definidas.')

## 3. Pool predictions over the 5 test folds

Each fold uses its own validation-tuned thresholds (from NB03's `m2_results.json`),
exactly as in the paper. We keep, per image: pathology probs, binary preds, ground
truth, and the artifact columns (SALIVA, LUZ).

In [5]:
with open(NB03_RESULTS, encoding='utf-8') as f:
    nb03 = json.load(f)

pooled = {'probs': [], 'preds': [], 'labels': [], 'SALIVA': [], 'LUZ': [], 'fold': []}

for fold in range(N_FOLDS):
    df_te = pd.read_csv(SPLITS_DIR / f'fold_{fold}_test.csv')
    model = load_fold_model(fold)
    probs = predict_fold(model, df_te)
    thr   = np.array(nb03[EXP_NAME]['fold_results'][fold]['val_thresholds'])
    preds = (probs >= thr).astype(int)

    pooled['probs'].append(probs)
    pooled['preds'].append(preds)
    pooled['labels'].append(df_te[CORE_LABELS].values.astype(int))
    for art in ARTIFACTS:
        pooled[art].append(df_te[art].values.astype(int))
    pooled['fold'].append(np.full(len(df_te), fold))
    print(f'  Fold {fold}: {len(df_te)} imagens preditas.')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

probs_all  = np.concatenate(pooled['probs'])
preds_all  = np.concatenate(pooled['preds'])
labels_all = np.concatenate(pooled['labels'])
artifact_all = {art: np.concatenate(pooled[art]) for art in ARTIFACTS}
print(f'\nTotal agrupado: {len(labels_all)} imagens (esperado ~1990).')
print(f'F1-macro agrupado: {f1_score(labels_all, preds_all, average="macro", zero_division=0):.4f}')

## 4. Shortcut audit — every artifact × pathology pair

In [6]:
def boot_rate(binary_vec, B=2000, seed=42):
    """CI95 de uma proporção via bootstrap (para a FP-rate)."""
    if len(binary_vec) == 0:
        return (float('nan'), float('nan'))
    rng = np.random.default_rng(seed)
    n = len(binary_vec)
    idx = rng.integers(0, n, size=(B, n))
    rates = binary_vec[idx].mean(axis=1)
    return (float(np.percentile(rates, 2.5)), float(np.percentile(rates, 97.5)))

audit = []
for art in ARTIFACTS:
    art_vec = artifact_all[art]
    for i, path in enumerate(CORE_LABELS):
        gt   = labels_all[:, i]
        pred = preds_all[:, i]

        n_path        = int(gt.sum())
        n_path_art    = int(((gt == 1) & (art_vec == 1)).sum())
        cooc          = n_path_art / n_path if n_path > 0 else float('nan')

        # F1 da patologia no subconjunto com / sem o artefato
        m_art = art_vec == 1
        m_no  = art_vec == 0
        f1_with = float(f1_score(gt[m_art], pred[m_art], average='binary', zero_division=0)) if m_art.sum() > 0 else float('nan')
        f1_wo   = float(f1_score(gt[m_no],  pred[m_no],  average='binary', zero_division=0)) if m_no.sum()  > 0 else float('nan')

        # FP-rate: artefato presente, patologia AUSENTE → modelo prediz patologia?
        m_artonly = (art_vec == 1) & (gt == 0)
        n_artonly = int(m_artonly.sum())
        fp_vec    = pred[m_artonly]
        fp_rate   = float(fp_vec.mean()) if n_artonly > 0 else float('nan')
        ci_low, ci_high = boot_rate(fp_vec) if n_artonly > 0 else (float('nan'), float('nan'))

        # baseline: FP-rate da mesma patologia SEM o artefato (referência)
        m_noart_neg = (art_vec == 0) & (gt == 0)
        fp_base = float(pred[m_noart_neg].mean()) if m_noart_neg.sum() > 0 else float('nan')

        # FN-rate: artefato presente, patologia PRESENTE → modelo prediz negativo?
        m_art_pos = (art_vec == 1) & (gt == 1)
        n_art_pos = int(m_art_pos.sum())
        fn_vec    = (pred[m_art_pos] == 0).astype(int)
        fn_rate   = float(fn_vec.mean()) if n_art_pos > 0 else float('nan')

        # baseline: FN-rate da mesma patologia SEM o artefato
        m_noart_pos = (art_vec == 0) & (gt == 1)
        fn_base = float((pred[m_noart_pos] == 0).mean()) if m_noart_pos.sum() > 0 else float('nan')

        audit.append({
            'artifact': art, 'pathology': path,
            'n_pathology': n_path, 'n_pathology_with_artifact': n_path_art,
            'cooccurrence': cooc,
            'f1_with_artifact': f1_with, 'f1_without_artifact': f1_wo,
            'f1_gap': (f1_with - f1_wo) if not (np.isnan(f1_with) or np.isnan(f1_wo)) else float('nan'),
            'n_artifact_only': n_artonly,
            'fp_rate_artifact_only': fp_rate,
            'fp_rate_ci_low': ci_low, 'fp_rate_ci_high': ci_high,
            'fp_rate_baseline_no_artifact': fp_base,
            'fp_excess_vs_baseline': (fp_rate - fp_base) if not (np.isnan(fp_rate) or np.isnan(fp_base)) else float('nan'),
            'n_artifact_and_pathology': n_art_pos,
            'fn_rate_with_artifact': fn_rate,
            'fn_rate_baseline_no_artifact': fn_base,
            'fn_excess_vs_baseline': (fn_rate - fn_base) if not (np.isnan(fn_rate) or np.isnan(fn_base)) else float('nan'),
        })

df_audit = pd.DataFrame(audit)
with open(RESULTS_DIR / 'shortcut_full.json', 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
df_audit.to_csv(RESULTS_DIR / 'shortcut_full.csv', index=False, encoding='utf-8-sig')

pd.set_option('display.width', 180, 'display.max_columns', 20)
print(df_audit[['artifact','pathology','n_pathology','cooccurrence',
                'f1_gap','n_artifact_only','fp_excess_vs_baseline',
                'n_artifact_and_pathology','fn_excess_vs_baseline']]
      .round(3).to_string(index=False))

## 5. Verdict — flag suspicious pairs

A pair is flagged as a **candidate shortcut** when the artifact-only FP-rate is both
high in absolute terms and clearly above the no-artifact baseline, with enough support
to trust it. Thresholds are conservative and reported explicitly (no silent cutoffs).

In [7]:
FP_ABS_THR    = 0.20   # FP-rate absoluta mínima p/ suspeita
FP_EXCESS_THR = 0.10  # excesso sobre o baseline sem artefato
FN_ABS_THR    = 0.20   # FN-rate absoluta mínima p/ suspeita
FN_EXCESS_THR = 0.10  # excesso de FN sobre o baseline sem artefato
MIN_SUPPORT   = 10     # nº mínimo de imagens p/ confiar

print('=== VEREDITO POR PAR (artefato → patologia) ===\n')
flagged = []
for r in audit:
    n_fp = r['n_artifact_only']
    fp = r['fp_rate_artifact_only']
    exc_fp = r['fp_excess_vs_baseline']
    
    n_fn = r['n_artifact_and_pathology']
    fn = r['fn_rate_with_artifact']
    exc_fn = r['fn_excess_vs_baseline']
    
    tag = f"{r['artifact']:>6} → {r['pathology']:<18}"
    
    suspicious_fp = (n_fp >= MIN_SUPPORT) and (not np.isnan(fp) and fp >= FP_ABS_THR) and (not np.isnan(exc_fp) and exc_fp >= FP_EXCESS_THR)
    suspicious_fn = (n_fn >= MIN_SUPPORT) and (not np.isnan(fn) and fn >= FN_ABS_THR) and (not np.isnan(exc_fn) and exc_fn >= FN_EXCESS_THR)
    
    if suspicious_fp or suspicious_fn:
        flagged.append(r)
        if suspicious_fp:
            print(f'  {tag}  ⚠ SHORTCUT FP (Falso Positivo) FP={fp:.1%} '
                  f'(excesso={exc_fp:+.1%}, n={n_fp})')
        if suspicious_fn:
            print(f'  {tag}  ⚠ SHORTCUT FN (Esconde Doença) FN={fn:.1%} '
                  f'(excesso={exc_fn:+.1%}, n={n_fn})')
    else:
        if n_fp < MIN_SUPPORT and n_fn < MIN_SUPPORT:
            print(f'  {tag}  suporte insuficiente (n_fp={n_fp}, n_fn={n_fn}) — inconclusivo')
        else:
            print(f'  {tag}  ✓ sem atalho dominante')

print(f'\nPares sinalizados: {len(flagged)}')
print('NOTA: pares com suporte < %d imagens são reportados como inconclusivos.' % MIN_SUPPORT)

with open(RESULTS_DIR / 'shortcut_flagged.json', 'w', encoding='utf-8') as f:
    json.dump({'flagged': flagged,
               'thresholds': {'fp_abs': FP_ABS_THR, 'fp_excess': FP_EXCESS_THR,
                              'fn_abs': FN_ABS_THR, 'fn_excess': FN_EXCESS_THR,
                              'min_support': MIN_SUPPORT}},
              f, ensure_ascii=False, indent=2)

## 6. Figures (EN + PT) — heatmaps of co-occurrence and FP-rate

In [8]:
def pivot(metric):
    return df_audit.pivot(index='artifact', columns='pathology', values=metric)[CORE_LABELS]

TXT = {
    'en': {'title': 'Shortcut audit — artifact × pathology (Swin-Tiny + CCR)',
           'cooc': 'Co-occurrence  P(artifact | pathology)',
           'fp':   'FP-rate (False Positive due to artifact)',
           'fn':   'FN-rate (Missed pathology due to artifact)'},
    'pt': {'title': 'Auditoria de atalhos — artefato × patologia (Swin-Tiny + CCR)',
           'cooc': 'Co-ocorrência  P(artefato | patologia)',
           'fp':   'Taxa FP (Falso Positivo gerado pelo artefato)',
           'fn':   'Taxa FN (Patologia ocultada pelo artefato)'},
}

def make_fig(lang):
    fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
    sns.heatmap(pivot('cooccurrence'), annot=True, fmt='.2f', cmap='Blues',
                vmin=0, vmax=0.6, linewidths=0.4, ax=axes[0], cbar_kws={'label': ''})
    axes[0].set_title(TXT[lang]['cooc'], fontweight='bold', fontsize=10)
    axes[0].set_xlabel(''); axes[0].set_ylabel('')
    
    sns.heatmap(pivot('fp_rate_artifact_only'), annot=True, fmt='.2f', cmap='Reds',
                vmin=0, vmax=0.5, linewidths=0.4, ax=axes[1], cbar_kws={'label': ''})
    axes[1].set_title(TXT[lang]['fp'], fontweight='bold', fontsize=10)
    axes[1].set_xlabel(''); axes[1].set_ylabel('')
    
    sns.heatmap(pivot('fn_rate_with_artifact'), annot=True, fmt='.2f', cmap='Oranges',
                vmin=0, vmax=0.5, linewidths=0.4, ax=axes[2], cbar_kws={'label': ''})
    axes[2].set_title(TXT[lang]['fn'], fontweight='bold', fontsize=10)
    axes[2].set_xlabel(''); axes[2].set_ylabel('')
    
    plt.suptitle(TXT[lang]['title'], fontweight='bold', y=1.04)
    plt.tight_layout()
    out = FIGS_DIR / f'nb07_shortcut_audit_{lang}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura salva: {out.name}')

for lang in ['en', 'pt']:
    make_fig(lang)

## 7. Paragraph for the paper

In [9]:
luz_ero = next(r for r in audit if r['artifact'] == 'LUZ' and r['pathology'] == 'EROSÃO')
n_pairs = len(audit)
n_flag  = len(flagged)
print(f"""
--- Sugestão de parágrafo (seção Shortcut audit) ---

We extended the artifact audit beyond the LIGHT–EROSION pair to all
{n_pairs} artifact×pathology combinations ({', '.join(ARTIFACTS)} × five findings),
reusing the trained Swin-Tiny+CCR models over the pooled five-fold test set
(n={len(labels_all)}). For each pair we measured co-occurrence, the pathology F1 with
versus without the artifact, and the false-positive rate on images where the artifact
is present but the pathology is absent. For LIGHT–EROSION, the co-occurrence was
{luz_ero['cooccurrence']:.1%} and the artifact-only false-positive rate was
{luz_ero['fp_rate_artifact_only']:.1%} (baseline without artifact:
{luz_ero['fp_rate_baseline_no_artifact']:.1%}). Across all pairs, {n_flag} exceeded the
pre-registered shortcut thresholds (FP-rate >= {0.20:.0%} and excess over baseline
>= {0.10:.0%} with >= 10 supporting images); pairs with insufficient support are reported
as inconclusive rather than clean.
""")